In [13]:
# %cd ../../..

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, TimeDistributed, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score
import numpy as np
import os
import pandas as pd

# Pobieranie i preprocesowanie datasetu

In [15]:
dataset_path = './MLPSD.pkl'

In [16]:
df = pd.read_pickle(dataset_path)

In [17]:
df

,annotation_id,annotator,created_at,errors_summary,id,lead_time,overall_summary,rating,squat_types,tricks,updated_at,video_url,fps,video_time,frames,labels,video_path,keypoints,distance_matrices_normalized,distance_matrices
0,13,1,2024-11-18T22:25:23.511525Z,NaN,366,698.367,NaN,7,diffrent,"[{'start': 13.374851720047449, 'end': 15.34460...",2024-11-18T22:25:23.511525Z,s3://squat-label-studio/squats/lifting/0-0084-...,30.0,41.0,1230.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0084-50-7-0-2-0-0.mp4,"[[[0.0, 0.0, 0.0], [0.008501114, -0.030775517,...","[[0.11267499, 0.4931849, 0.5039826, 0.10954244...","[[[0.0, 0.20962131, 0.7906997, 0.807189, 0.204..."
1,11,1,2024-11-18T22:08:33.297645Z,NaN,367,267.864,NaN,7,NaN,NaN,2024-11-18T22:09:00.052430Z,s3://squat-label-studio/squats/lifting/0-0085-...,30.0,14.0,420.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0085-80-8-1-2-0-0.mp4,"[[[0.0, 0.0, 0.0], [0.06892364, -0.0035920888,...","[[0.06751165, 0.41037306, 0.5385046, 0.0776159...","[[[0.0, 0.1513325, 0.6569368, 0.84588736, 0.16..."
2,12,1,2024-11-18T22:15:06.188387Z,NaN,368,413.600,Ćwiczenie zdecydowanie wykonywane zbyt szybko....,6,NaN,"[{'start': 1.0688018979833926, 'end': 1.804863...",2024-12-05T15:41:23.188734Z,s3://squat-label-studio/squats/lifting/0-0086-...,30.0,17.0,510.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0086-1-5-0-3-0-0.mp4,"[[[0.0, 0.0, 0.0], [-0.11756204, 0.045475494, ...","[[0.06228109, 0.34307784, 0.620119, 0.10509756...","[[[0.0, 0.13068436, 0.5295907, 0.9231618, 0.19..."
3,28,1,2024-12-05T15:43:49.471138Z,NaN,369,142.490,NaN,7,low_bar,NaN,2024-12-05T15:43:49.471138Z,s3://squat-label-studio/squats/lifting/0-0087-...,30.0,20.0,600.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0087-5-1-3-0-0.mp4,"[[[0.0, 0.0, 0.0], [-0.16868314, 0.0900087, -0...","[[0.115934655, 0.30324504, 0.57848823, 0.05246...","[[[0.0, 0.1936774, 0.48096266, 0.90311396, 0.0..."
4,29,1,2024-12-05T15:44:43.549323Z,NaN,371,30.621,Ćwiczenie zdecydowanie wykonywane zbyt szybko....,6,high_bar,NaN,2024-12-05T15:44:43.549323Z,s3://squat-label-studio/squats/lifting/0-0089-...,30.0,9.0,270.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0089-60-5-0-3-0-1.mp4,"[[[0.0, 0.0, 0.0], [-0.11497185, -0.0050474256...","[[0.04825226, 0.35160255, 0.49964395, 0.0, 0.3...","[[[0.0, 0.13721392, 0.51458657, 0.6987524, 0.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,169,1,2024-12-31T12:26:51.015675Z,NaN,695,74.509,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:26:51.015675Z,s3://squat-label-studio/squats/formcheck_text/...,30.0,27.0,810.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/formcheck_text/1-0020-100-5-1-3-1-0.mp4,"[[[0.0, 0.0, 0.0], [-0.115019605, 0.0445962, -...","[[0.02192016, 0.26873413, 0.5659629, 0.0221559...","[[[0.0, 0.123797774, 0.50011075, 0.95329034, 0..."
219,168,1,2024-12-31T12:24:56.342462Z,NaN,698,180.816,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:24:56.342462Z,s3://squat-label-studio/squats/formcheck_text/...,30.0,75.0,2250.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/formcheck_text/1-0023-1-5-1-3-1-0.mp4,"[[[0.0, 0.0, 0.0], [-0.13167235, -0.015190777,...","[[0.014085502, 0.24669935, 0.5428832, 0.011459...","[[[0.0, 0.13257366, 0.47650826, 0.9144352, 0.1..."
220,167,1,2024-12-31T12:21:54.353300Z,NaN,699,51.724,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:21:54.353300Z,s3://squat-label-studio/squats/formcheck_text/...,30.0,33.0,990.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/formcheck_text/1-0024-1-6-1-3-1-0.mp4,"[[[0.0, 0.0, 0.0], [-0.13210404, -0.012553379,...","[[0.019306833, 0.2554605, 0.5482265, 0.0170570...","[[[0.0, 0.13354048, 0.48112205, 0.91202813, 0...."
221,166,1,2024-12-31T12:21:00.532898Z,NaN,700,26.237,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:21:00.532898Z,s3:/

In [18]:
df = df[df['distance_matrices'].notna()]
df['distance_matrices_normalized'] = df['distance_matrices_normalized'].apply(lambda x: np.array(x) if isinstance(x, list) else x)

In [19]:
df

,annotation_id,annotator,created_at,errors_summary,id,lead_time,overall_summary,rating,squat_types,tricks,updated_at,video_url,fps,video_time,frames,labels,video_path,keypoints,distance_matrices_normalized,distance_matrices
0,13,1,2024-11-18T22:25:23.511525Z,NaN,366,698.367,NaN,7,diffrent,"[{'start': 13.374851720047449, 'end': 15.34460...",2024-11-18T22:25:23.511525Z,s3://squat-label-studio/squats/lifting/0-0084-...,30.0,41.0,1230.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0084-50-7-0-2-0-0.mp4,"[[[0.0, 0.0, 0.0], [0.008501114, -0.030775517,...","[[0.11267499, 0.4931849, 0.5039826, 0.10954244...","[[[0.0, 0.20962131, 0.7906997, 0.807189, 0.204..."
1,11,1,2024-11-18T22:08:33.297645Z,NaN,367,267.864,NaN,7,NaN,NaN,2024-11-18T22:09:00.052430Z,s3://squat-label-studio/squats/lifting/0-0085-...,30.0,14.0,420.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0085-80-8-1-2-0-0.mp4,"[[[0.0, 0.0, 0.0], [0.06892364, -0.0035920888,...","[[0.06751165, 0.41037306, 0.5385046, 0.0776159...","[[[0.0, 0.1513325, 0.6569368, 0.84588736, 0.16..."
2,12,1,2024-11-18T22:15:06.188387Z,NaN,368,413.600,Ćwiczenie zdecydowanie wykonywane zbyt szybko....,6,NaN,"[{'start': 1.0688018979833926, 'end': 1.804863...",2024-12-05T15:41:23.188734Z,s3://squat-label-studio/squats/lifting/0-0086-...,30.0,17.0,510.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0086-1-5-0-3-0-0.mp4,"[[[0.0, 0.0, 0.0], [-0.11756204, 0.045475494, ...","[[0.06228109, 0.34307784, 0.620119, 0.10509756...","[[[0.0, 0.13068436, 0.5295907, 0.9231618, 0.19..."
3,28,1,2024-12-05T15:43:49.471138Z,NaN,369,142.490,NaN,7,low_bar,NaN,2024-12-05T15:43:49.471138Z,s3://squat-label-studio/squats/lifting/0-0087-...,30.0,20.0,600.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0087-5-1-3-0-0.mp4,"[[[0.0, 0.0, 0.0], [-0.16868314, 0.0900087, -0...","[[0.115934655, 0.30324504, 0.57848823, 0.05246...","[[[0.0, 0.1936774, 0.48096266, 0.90311396, 0.0..."
4,29,1,2024-12-05T15:44:43.549323Z,NaN,371,30.621,Ćwiczenie zdecydowanie wykonywane zbyt szybko....,6,high_bar,NaN,2024-12-05T15:44:43.549323Z,s3://squat-label-studio/squats/lifting/0-0089-...,30.0,9.0,270.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/lifting/0-0089-60-5-0-3-0-1.mp4,"[[[0.0, 0.0, 0.0], [-0.11497185, -0.0050474256...","[[0.04825226, 0.35160255, 0.49964395, 0.0, 0.3...","[[[0.0, 0.13721392, 0.51458657, 0.6987524, 0.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,169,1,2024-12-31T12:26:51.015675Z,NaN,695,74.509,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:26:51.015675Z,s3://squat-label-studio/squats/formcheck_text/...,30.0,27.0,810.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/formcheck_text/1-0020-100-5-1-3-1-0.mp4,"[[[0.0, 0.0, 0.0], [-0.115019605, 0.0445962, -...","[[0.02192016, 0.26873413, 0.5659629, 0.0221559...","[[[0.0, 0.123797774, 0.50011075, 0.95329034, 0..."
219,168,1,2024-12-31T12:24:56.342462Z,NaN,698,180.816,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:24:56.342462Z,s3://squat-label-studio/squats/formcheck_text/...,30.0,75.0,2250.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/formcheck_text/1-0023-1-5-1-3-1-0.mp4,"[[[0.0, 0.0, 0.0], [-0.13167235, -0.015190777,...","[[0.014085502, 0.24669935, 0.5428832, 0.011459...","[[[0.0, 0.13257366, 0.47650826, 0.9144352, 0.1..."
220,167,1,2024-12-31T12:21:54.353300Z,NaN,699,51.724,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:21:54.353300Z,s3://squat-label-studio/squats/formcheck_text/...,30.0,33.0,990.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",squats/formcheck_text/1-0024-1-6-1-3-1-0.mp4,"[[[0.0, 0.0, 0.0], [-0.13210404, -0.012553379,...","[[0.019306833, 0.2554605, 0.5482265, 0.0170570...","[[[0.0, 0.13354048, 0.48112205, 0.91202813, 0...."
221,166,1,2024-12-31T12:21:00.532898Z,NaN,700,26.237,Poprawnie wykonane ćwiczenie.,7,low_bar,NaN,2024-12-31T12:21:00.532898Z,s3:/

In [20]:
y = np.array(df['labels'])


In [21]:
y = [np.transpose(np.array(ele)) for ele in y]

In [22]:
X = np.array(df['distance_matrices_normalized'])

In [23]:
for x in X:
    print(x.shape)
    break

for ele in y:
    print(ele.shape)
    break

(1230, 136)
(1230, 10)


In [24]:
import numpy as np
np.savez('data.npz', X=X, y=y, allow_pickle=True)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (223,) + inhomogeneous part.

In [25]:
import pickle
with open('data.pkl', 'wb') as f:
    pickle.dump({'X': X, 'y': y}, f)

In [12]:
from sklearn.model_selection import train_test_split

### Padding

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
[seq.reshape(seq.shape[0], -1).shape for seq in X]

[(1440, 136),
 (600, 136),
 (360, 136),
 (450, 136),
 (1380, 136),
 (960, 136),
 (990, 136),
 (179, 136),
 (510, 136),
 (1379, 136),
 (810, 136),
 (1380, 136),
 (540, 136),
 (570, 136),
 (870, 136),
 (810, 136),
 (1170, 136),
 (420, 136),
 (390, 136),
 (810, 136),
 (1260, 136),
 (1170, 136),
 (810, 136),
 (930, 136),
 (1110, 136),
 (630, 136),
 (660, 136),
 (1050, 136),
 (660, 136),
 (720, 136),
 (1500, 136),
 (330, 136),
 (390, 136),
 (960, 136),
 (930, 136),
 (1590, 136),
 (480, 136),
 (570, 136),
 (540, 136),
 (570, 136),
 (690, 136),
 (1140, 136),
 (720, 136),
 (660, 136),
 (450, 136),
 (450, 136),
 (810, 136),
 (1440, 136),
 (1320, 136),
 (1050, 136),
 (780, 136),
 (510, 136),
 (690, 136),
 (510, 136),
 (840, 136),
 (510, 136),
 (870, 136),
 (870, 136),
 (1200, 136),
 (570, 136),
 (630, 136),
 (870, 136),
 (570, 136),
 (480, 136),
 (600, 136),
 (960, 136),
 (1050, 136),
 (1080, 136),
 (510, 136),
 (810, 136),
 (420, 136)]

In [15]:
# Padding dla X i y
max_len = max([seq.shape[0] for seq in X])  # Maksymalna długość sekwencji
X = pad_sequences([seq.reshape(seq.shape[0], -1) for seq in X], maxlen=max_len, padding='post', dtype='float32').reshape(len(X), max_len, 136)
y = pad_sequences(y, maxlen=max_len, padding='post', dtype='int')

# Podział datasetu

In [16]:
# Podział na zbiór treningowy (90%) i testowy (10%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Podział zbioru treningowego na treningowy (70% z całych danych) i walidacyjny (10% z całych danych)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.125, random_state=42)  # 0.125 * 80% = 10%

# Tworzenie modelu BI-LSTM

In [32]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, TimeDistributed, Masking
from tensorflow.keras.layers import Lambda

@tf.function
def model_call(input_data):
    return model(input_data)

tf.config.run_functions_eagerly(True)

In [33]:
# Rozmiary wejściowe
input_size = 136  # Rozmiar wektora dystansów
n_classes = 10

In [34]:
def define_model():
    # Definicja modelu
    input_layer = Input(shape=(None, input_size))  # Sekwencja o dowolnej długości
    x = Masking(mask_value=0.0)(input_layer)  # Maskowanie wartości zerowych
    x = Bidirectional(LSTM(64, return_sequences=True))(x)  # Dwukierunkowa LSTM
    x = Bidirectional(LSTM(64, return_sequences=True))(x)

    output_layer = TimeDistributed(Dense(n_classes, activation='sigmoid'))(x)  # Wynik dla każdej klatki
    
    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    model.summary()
    return model

In [35]:
model = define_model()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)           ┃ Output Shape         ┃     Param # ┃ Connected to         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3          │ (None, None, 136)    │           0 │ -                    │
│ (InputLayer)           │                      │             │                      │
├────────────────────────┼──────────────────────┼─────────────┼──────────────────────┤
│ not_equal_3 (NotEqual) │ (None, None, 136)    │           0 │ input_layer_3[0][0]  │
├────────────────────────┼──────────────────────┼─────────────┼──────────────────────┤
│ masking_3 (Masking)    │ (None, None, 136)    │           0 │ input_layer_3[0][0]  │
├────────────────────────┼──────────────────────┼─────────────┼──────────────────────┤
│ any_3 (Any)            │ (None, None)         │           0 │ not_equal_3[0][0]    │
├────────────────────────┼──────────────────────┼─────────────┼──────────────────────┤
│ bidirectional_6        │ (None, None, 128)    │     102,912 │ masking_3[0][0],     │
│ (Bidirectional)        │                      │             │ any_3[0][0]          │
├────────────────────────┼──────────────────────┼─────────────┼──────────────────────┤
│ bidirectional_7        │ (None, None, 128)    │      98,816 │ bidirectional_6[0][… │
│ (Bidirectional)        │                      │             │ any_3[0][0]          │
├────────────────────────┼──────────────────────┼─────────────┼──────────────────────┤
│ time_distributed_2     │ (None, None, 10)     │       1,290 │ bidirectional_7[0][… │
│ (TimeDistributed)      │                      │             │ any_3[0][0]          │
└────────────────────────┴──────────────────────┴─────────────┴──────────────────────┘

 Total params: 203,018 (793.04 KB)

 Trainable params: 203,018 (793.04 KB)

 Non-trainable params: 0 (0.00 B)

# Trenowanie

In [ ]:
# Trening modelu
model.fit(X_train, y_train, batch_size=8, epochs=5)

Epoch 1/10


C:\Users\kubak\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 220s 53s/step - accuracy: 0.3900 - loss: 0.6283
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 200s 49s/step - accuracy: 0.0000e+00 - loss: 0.2782
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 201s 50s/step - accuracy: 0.0000e+00 - loss: 0.1439
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 218s 54s/step - accuracy: 0.0000e+00 - loss: 0.0830
Epoch 5/10
2/4 ━━━━━━━━━━━━━━━━━━━━ 1:48 54s/step - accuracy: 2.9481e-05 - loss: 0.0576